In [16]:
import torch


loaded_tensor = torch.load('/home/hyukjun/Projects/MonoGS_git/Results/Pixel_data/fp32.pt')

def float32_to_custom_float16(tensor):
    # Ensure tensor is a float32 type
    tensor = tensor.to(torch.float32)
    
    # Create a tensor to hold the custom 16-bit float format
    custom_fp16_tensor = torch.zeros_like(tensor, dtype=torch.int16)

    # Extract the sign, exponent, and mantissa from float32 using bitwise operations
    float32_bits = tensor.view(torch.int32)  # View as int32 to access raw bits

    # Mask and shift to extract the sign, exponent, and mantissa
    sign = (float32_bits >> 31) & 0x1       # Extract sign (1 bit)
    exponent = (float32_bits >> 23) & 0xFF  # Extract exponent (8 bits)
    mantissa = float32_bits & 0x7FFFFF      # Extract mantissa (23 bits)

    # Adjust mantissa to fit 7 bits (custom format)
    # Right shift mantissa to keep only 7 bits
    adjusted_mantissa = mantissa >> (23 - 7)

    # Combine sign, exponent, and mantissa to form 16-bit custom float
    custom_fp16 = (sign << 15) | (exponent << 7) | adjusted_mantissa
    custom_fp16_tensor.copy_(custom_fp16)

    return custom_fp16_tensor

def print_tensor_bitwise(tensor):
    # Convert each element to its binary representation
    for i in range(tensor.numel()):  # Iterate over each element in the tensor
        binary_str = format(tensor.view(-1)[i].item(), '016b')  # Convert to 16-bit binary string
        print(f"Element {i}: {binary_str}")

# # Example usage:
# tensor = torch.tensor([1.5, -2.25, 3.75], dtype=torch.float32)
# custom_fp16_tensor = float32_to_custom_float16(tensor)

# # Print tensor in bitwise format
# print("Custom FP16 tensor in bitwise format:")
# print_tensor_bitwise(custom_fp16_tensor)


fp16_tensor = {}

# Assuming loaded_tensor is a dictionary
for key, value in loaded_tensor.items():
    fp16_tensor[key] = float32_to_custom_float16(value)
    
torch.save(fp16_tensor, '/home/hyukjun/Projects/MonoGS_git/Results/Pixel_data/fp16.pt')
